# 001 - Level 1 

## 1. Data Exploration

### 🔎 Operaciones básicas

- **SELECT**  
  Recupera todos los registros de una tabla.

- **COUNT()**  
  Cuenta el **número total de filas**.

- **DISTINCT**  
  Elimina valores duplicados y permite ver **registros únicos**.


In [2]:
import pandas as pd
import numpy as np
import polars as pl

In [60]:
df_matches = pd.read_csv('../Data/WorldCupMatches.csv').rename(columns=lambda col: col.lower())
df_players = pd.read_csv('../Data/WorldCupPlayers.csv').rename(columns=lambda col: col.lower())
df_cups = pd.read_csv('../Data/WorldCups.csv').rename(columns=lambda col: col.lower())

pl_matches = pl.read_csv('../Data/WorldCupMatches.csv').rename(lambda col: col.lower())
pl_players = pl.read_csv('../Data/WorldCupPlayers.csv').rename(lambda col: col.lower())
pl_cups = pl.read_csv('../Data/WorldCups.csv').rename(lambda col: col.lower())

## 🏆 Ejercicio 1.1: Auditoría de Datos Mundialistas

**Nivel:** 1 (Exploración)

### 🎯 Objetivo
Determinar el volumen total de datos, cuántos años únicos cubre el dataset y visualizar la estructura básica.

### 📋 El Reto
Necesitamos responder tres preguntas rápidas sobre nuestra tabla de partidos (`matches`):

1. ¿Cuántos registros totales hay?
2. ¿Cuántos mundiales (años) diferentes están registrados?
3. ¿Cómo se ven los primeros 5 registros para entender el formato?


```SQL
SELECT
    *,
    COUNT(*) OVER() AS total_registros,
    (SELECT COUNT(DISTINCT year) FROM worldcup.matches) AS total_mundiales
FROM worldcup.matches
LIMIT 5;
```

In [15]:
df_m = df_matches.copy()

df_m['total_registros'] = len(df_m['year'])

df_m['total_mundiales'] = df_m['year'].nunique()

In [21]:
res = pl_matches.with_columns([
    pl.len().alias('total_registros'),
    pl.col('year').drop_nulls().n_unique().alias('total_mundiales')
])

## 🛠️ Ejercicio 1.2: Selección y Limpieza de Nulos

En este ejercicio vamos a simular que recibimos el dataset **“sucio”** y queremos dejarlo listo para trabajar.

### 🎯 Objetivo

- Seleccionar únicamente las siguientes columnas:
  - `year`
  - `home_team_name`
  - `home_team_goals`
  - `away_team_goals`
  - `away_team_name`
- Identificar si existen filas con valores nulos en la columna `year`.
- Eliminar las filas donde `year` sea nulo.
- Cambiar el tipo de dato de la columna `year` a **entero**, ya que en algunos casos puede venir como decimal o texto.

```SQL
WITH Auditoria AS(
    SELECT
        year,
        "Home Team Name",
        "Home Team Goals",
        "Away Team Name",
        "Away Team Goals",
        SUM(CASE WHEN year is NULL THEN 1 ELSE 0 END) OVER() AS total_nulos_encontrados
    FROM worldcup.matches
)
SELECT
    year::INTEGER,
    "Home Team Name",
    "Home Team Goals",
    "Away Team Name",
    "Away Team Goals",
    total_nulos_encontrados
FROM Auditoria
WHERE year IS NOT NULL;
```

In [25]:
df_m = df_matches.copy()

total_nulos = df_m['year'].isna().sum()

df_clean = df_m.dropna(subset=['year']).copy()
df_clean['year'] = df_clean['year'].astype(int)

df_clean['total_nulos_encontrados'] = total_nulos

In [28]:
res = (
    pl_matches.with_columns([
        pl.col('year').null_count().alias('total_nulos_encontrados')
    ])
    .filter(
        pl.col('year').is_not_null()
    )
    .with_columns([
        pl.col('year').cast(pl.Int32, strict=True)
    ])
    .select([
        'year',
        'home team name',
        'home team goals',
        'away team name', 
        'away team goals',
        'total_nulos_encontrados'
    ])
)

## 🏋️ Desafío 1.A: El Quirófano de Columnas

A veces recibimos columnas con nombres horribles (espacios, mayúsculas locas, puntos).  
Para trabajar bien, necesitamos **estandarizar**.

### 🎯 Objetivo

- Seleccionar únicamente las columnas:
  - `year`
  - `home_team_name`
  - `away_team_name`
- Renombrar las columnas a:
  - `anio`
  - `local`
  - `visitante`
- Mostrar únicamente las **últimas 3 filas** del resultado.


In [42]:
df_m = df_matches[['year','home team name', 'away team name']].copy()

df_m = df_m[['year', 'home team name', 'away team name']].rename(
    columns={'year':'anio', 'home team name':'local', 'away team name':'visitante'}
).tail(3).reset_index(drop=True)

#res = df_m.sort_values(by='anio', ascending=False).head(3).reset_index(drop=True)

res

,anio,local,visitante
0,2014.0,Argentina,Switzerland
1,2014.0,Germany,Argentina
2,2014.0,Brazil,Netherlands


In [57]:
res_pl = (
    pl_matches
    .select([
        pl.col("year").alias("anio"),
        pl.col("home team name").alias("local"),
        pl.col("away team name").alias("visitante")
    ])
    .drop_nulls()
    .tail(3)
    .with_row_index(name='indice')
)

## 🏋️ Desafío 1.B: El Filtro de Unicidad (Duplicados)

En los datos de la Copa del Mundo, a veces un partido se registra dos veces por error.  
Antes de cualquier análisis serio, es fundamental validar la unicidad de los registros.

---

### 🎯 Objetivo

Detectar y eliminar registros duplicados para asegurar la calidad del dataset.

---

### 📌 Tareas del desafío

- Contar cuántas **filas totales** tiene el dataset original.
- Eliminar las filas que estén **totalmente duplicadas**  
  (donde todas las columnas tengan exactamente los mismos valores).
- Calcular **cuántas filas se eliminaron**, mostrando la **diferencia** entre el total original y el dataset limpio.

---

Este paso es clave para evitar distorsiones en conteos, agregaciones y métricas posteriores.


In [60]:
total_inicial_pd = len(df_matches)

df_unico_pd = df_matches.drop_duplicates()

total_final_pd = len(df_unico_pd)
eliminados_pd = total_inicial_pd - total_final_pd

print(f"Total original: {total_inicial_pd}")
print(f"Filas eliminadas: {eliminados_pd}")

Total original: 4572
Filas eliminadas: 3735


In [61]:
total_inicial_pl = pl_matches.height

df_unico_pl = pl_matches.unique()

total_final_pl = df_unico_pl.height
eliminados_pl = total_inicial_pl - total_final_pl

print(f"Total original: {total_inicial_pl}")
print(f"Filas eliminadas: {eliminados_pl}")

Total original: 4572
Filas eliminadas: 3735


In [27]:
home = df_matches[['home team name', 'home team goals']].rename(columns={'home team name':'teams', 'home team goals': 'goals'})
away = df_matches[['away team name', 'away team goals']].rename(columns={'away team name':'teams', 'away team goals': 'goals'})

equipos = pd.concat([home, away], axis=0)

res_pd = (
    equipos[equipos['teams'].notnull()]    
    .groupby('teams')['goals']
    .sum()
    .reset_index()
    .rename(columns={'goals':'total_goals'})
    .sort_values(by='teams')
)

In [30]:
res_pl = (
    pl.concat([
        pl_matches.select([
            pl.col('home team name').alias('team'),
            pl.col('home team goals').alias('goals')
        ]),
        pl_matches.select([
            pl.col('away team name').alias('team'),
            pl.col('away team goals').alias('goals')
        ])
    ])
    .drop_nulls(subset=['team'])
    .group_by('team')
    .agg(
        pl.col('goals').sum().alias('total_goals')
    )
    .sort('team')
)

```SQL
SELECT
    year,
    "Home Team Name",
    "Away Team Name",
    "Home Team Goals",
    "Away Team Goals",
    (COALESCE("Home Team Goals",0) + COALESCE("Away Team Goals",0)) AS total_score
FROM worldcup.matches
WHERE "Home Team Name" IS NOT NULL AND "Away Team Name" IS NOT NULL
ORDER BY total_score DESC;
```

In [35]:
import pandas as pd

res_pd = (
    df_matches
    # subset indica qué columnas revisar para buscar nulos
    .dropna(subset=["home team goals", "away team goals"])
    .assign(total_score = lambda x: x["home team goals"] + x["away team goals"])
    .sort_values(by="total_score", ascending=False)
)

In [38]:
res_pl = (
    pl_matches
    # Elimina la fila si hay nulo en cualquiera de estas dos columnas
    .drop_nulls(subset=["home team goals", "away team goals"])
    .with_columns(
        (pl.col("home team goals") + pl.col("away team goals")).alias("total_score")
    )
    .sort("total_score", descending=True)
)

In [40]:
def sumar_diez(n):
    return n + 10

res = sumar_diez(5)

In [42]:
sumar_diez_rapido = lambda n : n + 10

res = sumar_diez_rapido(5)

```SQL
WITH goles_totales AS (
    SELECT year,
          "Home Team Name",
          "Away Team Name",
          "Home Team Goals",
          "Away Team Goals",
          (COALESCE("Home Team Goals", 0) + COALESCE("Away Team Goals", 0)) AS total_score,
          RANK() OVER(PARTITION BY year ORDER BY (COALESCE("Home Team Goals", 0) + COALESCE("Away Team Goals", 0)) DESC) as ranking
   FROM worldcup.matches
   WHERE "Home Team Name" IS NOT NULL
     AND "Away Team Name" IS NOT NULL
)
SELECT
    year,
    "Home Team Name",
    "Away Team Name",
    "Home Team Goals",
    "Away Team Goals",
    total_score
FROM goles_totales
WHERE ranking = 1
ORDER BY total_score DESC;
```

In [50]:
df = df_matches.dropna(subset=['home team name']).copy()
df['total_score'] = df['home team goals'].fillna(0) + df['away team goals'].fillna(0)

max_por_año = df.groupby('year')['total_score'].transform('max')

res_pd = df[df['total_score'] == max_por_año].sort_values('total_score', ascending=False)

res_pd = res_pd[['year', 'home team name', 'away team name', 'home team goals', 'away team goals', 'total_score']]

In [47]:
res_pl = (
    pl_matches
    .drop_nulls(subset=['home team name'])
    .with_columns(
        total_score = pl.col('home team goals').fill_null(0) + pl.col('away team goals').fill_null(0)
    )
    .with_columns(
        ranking = pl.col('total_score').rank('dense', descending=True).over('year')
    )
    .filter(pl.col('ranking') == 1)
    .sort('total_score', descending=True)
).select([
    'year','home team name', 'away team name', 'home team goals', 'away team goals', 'total_score'
])

```SQL
SELECT
    winner,
    COUNT(winner) AS total_cups,
    STRING_AGG(year::text, ', ' ORDER BY year ASC) AS years_won
FROM worldcup.cups
group by winner
ORDER BY total_cups DESC;
```

In [53]:
res_pd = (
    df_cups.groupby('winner')['year']
    .agg([
        ('total_cups', 'count'),
        ('year_won', lambda x: ', '.join(x.sort_values().astype(str)))
    ])
    .reset_index()
    .sort_values('total_cups', ascending=False)
)

In [55]:
res_pl = (
    pl_cups
    .group_by('winner')
    .agg([
        pl.len().alias('total_cups'),
        pl.col('year').sort().cast(pl.Utf8).str.join(', ').alias('years_won')
    ])
    .sort('total_cups', descending=True)
)

## 🧹 Checklist de Limpieza de Datos

| Orden | Fase           | Objetivo                                                     | Herramienta Clave              |
|------:|----------------|--------------------------------------------------------------|--------------------------------|
| 1     | Estructura     | Eliminar duplicados para no inflar métricas.                | `DISTINCT` / `unique()`        |
| 2     | Vacíos         | Resolver valores nulos (borrar fila o usar valor por defecto). | `COALESCE` / `fillna()`        |
| 3     | Formato Texto  | Quitar espacios fantasma y unificar mayúsculas/minúsculas.  | `TRIM` / `.str.strip()`        |
| 4     | Identidad      | Unificar nombres o alias (ej: `"Germany FR"` → `"Germany"`). | `CASE WHEN` / `replace()`     |
| 5     | Tipos          | Asegurar que números y fechas tengan el tipo correcto.      | `CAST` / `astype()`            |
| 6     | Lógica         | Sanity check: valores imposibles (goles negativos, años futuros). | `WHERE` / `filter()`      |
| 7     | Codificación   | Corregir caracteres extraños (ej: `Mxico` → `México`).       | `UTF-8` / `replace()`          |

In [70]:
df_m = df_matches.dropna(how='all').copy()

df_m['year'] = df_m['year'].astype(int)
# df_m['attendance'] = df_m['attendance'].astype(int)
df_m['attendance'] = df_m['attendance'].astype('Int64')

df_m.info()

<class 'pandas.core.frame.DataFrame'>
Index: 852 entries, 0 to 851
Data columns (total 20 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   year                  852 non-null    int64  
 1   datetime              852 non-null    object 
 2   stage                 852 non-null    object 
 3   stadium               852 non-null    object 
 4   city                  852 non-null    object 
 5   home team name        852 non-null    object 
 6   home team goals       852 non-null    float64
 7   away team goals       852 non-null    float64
 8   away team name        852 non-null    object 
 9   win conditions        852 non-null    object 
 10  attendance            850 non-null    Int64  
 11  half-time home goals  852 non-null    float64
 12  half-time away goals  852 non-null    float64
 13  referee               852 non-null    object 
 14  assistant 1           852 non-null    object 
 15  assistant 2           852 no

In [59]:
pl_matches.schema

Schema([('year', Int64),
        ('datetime', String),
        ('stage', String),
        ('stadium', String),
        ('city', String),
        ('home team name', String),
        ('home team goals', Int64),
        ('away team goals', Int64),
        ('away team name', String),
        ('win conditions', String),
        ('attendance', Int64),
        ('half-time home goals', Int64),
        ('half-time away goals', Int64),
        ('referee', String),
        ('assistant 1', String),
        ('assistant 2', String),
        ('roundid', Int64),
        ('matchid', Int64),
        ('home team initials', String),
        ('away team initials', String)])